# A/B Testing: Conversion, Effect Size and Statistical Uncertainty

**Type:** Original supplementary practice  
**Data:** Simulated independent user-level experiment

This notebook demonstrates a two-group conversion comparison. The analysis assumes independent random assignment and one observation per user. In a real experiment, verify randomisation, exposure logging, sample-ratio balance, guardrail metrics and the analysis plan before interpreting a p-value.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats

rng = np.random.default_rng(202)
n_control, n_treatment = 5000, 5050
p_control, p_treatment = .102, .116
control = rng.binomial(1, p_control, n_control)
treatment = rng.binomial(1, p_treatment, n_treatment)

summary = pd.DataFrame({
    "group":["Control","Treatment"],
    "users":[n_control,n_treatment],
    "conversions":[control.sum(),treatment.sum()],
})
summary["conversion_rate"] = summary["conversions"] / summary["users"]
display(summary)

## 1. State the hypotheses

- Null hypothesis: control and treatment conversion probabilities are equal.
- Alternative hypothesis: the conversion probabilities differ.
- Significance level: 5%, selected before looking at the result.

A two-sided test is used because either an increase or a decrease could matter.

In [ ]:
counts = np.array([control.sum(), treatment.sum()])
nobs = np.array([len(control), len(treatment)])
z_stat, p_value = proportions_ztest(count=counts, nobs=nobs, alternative="two-sided")

p0 = control.mean()
p1 = treatment.mean()
absolute_lift = p1 - p0
relative_lift = absolute_lift / p0
pooled = (control.sum() + treatment.sum()) / (len(control) + len(treatment))
se_null = np.sqrt(pooled * (1-pooled) * (1/len(control) + 1/len(treatment)))
ci_se = np.sqrt(p0*(1-p0)/len(control) + p1*(1-p1)/len(treatment))
ci = (absolute_lift - 1.96*ci_se, absolute_lift + 1.96*ci_se)

print(f"Control conversion:   {p0:.3%}")
print(f"Treatment conversion: {p1:.3%}")
print(f"Absolute lift:        {absolute_lift:.3%}")
print(f"Relative lift:        {relative_lift:.1%}")
print(f"z statistic:          {z_stat:.3f}")
print(f"p-value:              {p_value:.5f}")
print(f"Approx. 95% CI for absolute lift: [{ci[0]:.3%}, {ci[1]:.3%}]")

## 2. Interpretation

The p-value measures how unusual the observed difference (or a more extreme one) would be under the null model. It is not the probability that the null hypothesis is true, and statistical significance does not automatically imply business significance.

The confidence interval above is a large-sample approximation for the difference in two independent proportions.

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("The result is statistically significant at the 5% level.")
else:
    print("The result is not statistically significant at the 5% level.")

print("Decision context:")
print("- Is the confidence interval compatible with a practically meaningful lift?")
print("- Were users randomised and analysed at the correct unit?")
print("- Were the metric and stopping rule specified in advance?")
print("- Were guardrail metrics (refunds, latency, complaints) checked?")

## 3. Optional continuous outcome: Welch's t-test

Welch's t-test does not assume equal population variances. This example compares a synthetic continuous outcome, but the test choice must match the experimental design and estimand.

In [ ]:
control_value = rng.normal(250, 90, 1200)
treatment_value = rng.normal(263, 105, 1180)
t_stat, t_p = stats.ttest_ind(treatment_value, control_value, equal_var=False)
print("Control mean:", round(control_value.mean(),2))
print("Treatment mean:", round(treatment_value.mean(),2))
print("Welch t statistic:", round(t_stat,3))
print("p-value:", round(t_p,5))

## Limitations and extension

This is a simulation, not a real product experiment. For a real analysis, consider clustering or repeated observations, multiple testing, sequential monitoring, non-compliance, missing outcomes, novelty effects, and power/sample-size planning. A statistically significant result cannot repair a biased experiment.